In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 12, The Schouten–Nijenhuis bracket

Companion notebook to [12_schouten_nijenhuis.md](12_schouten_nijenhuis.md). `[·,·]_SN` extends the Lie bracket to multivector fields with the shifted grading `|X| = k − 1`, and supplies the universal Poisson obstruction `[π, π]_SN`.

## Shifted grading

Functions are SN-degree `−1`, vector fields are SN-degree `0`, bivectors are SN-degree `1`. Use `Graded(degree=...)` to declare these, **not `Scalar()`**, which the SN engine reads as `degree=0` (i.e. a 1-vector).

In [ ]:
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry

reg = PropertyRegistry()
X = Symbol("X"); reg.declare(X, Graded(degree=0))
Y = Symbol("Y"); reg.declare(Y, Graded(degree=0))
f = Symbol("f"); reg.declare(f, Graded(degree=-1))
g = Symbol("g"); reg.declare(g, Graded(degree=-1))

## Four base cases

`sn.expand(a, b, registry)` returns the closed form when both operands are atomic and SN-degrees are concrete integers in {−1, 0}. These are the characterising rules:

* `[X, Y]_SN = X*Y − Y*X`, the Lie bracket on 1-vectors.
* `[f, g]_SN = 0`, `C^∞` is commutative.
* `[f, X]_SN = −X(f)`, graded antisymmetric of the next.
* `[X, f]_SN = X(f)`, vectors act as derivations on functions.

In [ ]:
from jacopy.brackets.schouten import sn

print("[X, Y]_SN =", sn.expand(X, Y, reg))
print("[f, g]_SN =", sn.expand(f, g, reg))
print("[f, X]_SN =", sn.expand(f, X, reg))
print("[X, f]_SN =", sn.expand(X, f, reg))

## Wedge Leibniz

On `Product(X, Y)` (= `X ∧ Y`) the bracket descends via

```
[X ∧ Y, Z]_SN = X ∧ [Y, Z]_SN + (−1)^{|Y||Z|} [X, Z]_SN ∧ Y
```

so a 2-vector against a function reads as `X ∧ Y(f) + X(f) ∧ Y` (both signs `+1` because `|Y|·|f| = 0·(−1) = 0` is even).

In [ ]:
from jacopy.core.expr import Product

bivec = Product(X, Y)              # X ∧ Y
print("[X ∧ Y, f]_SN =", sn.expand(bivec, f, reg))

## Atomic higher-order multivectors stay opaque

A bare `Symbol` declared `Graded(degree=1)` plays the role of an atomic bivector, there's no wedge to peel, so SN can't descend. Rather than raise, `expand` returns the inert `BracketApply` node. That handle is exactly what makes `[π, π]_SN = 0` a usable hypothesis.

In [ ]:
pi = Symbol("π"); reg.declare(pi, Graded(degree=1))
obstr = sn.self_bracket(pi, reg)
print("[π, π]_SN =", obstr)
print("type      :", type(obstr).__name__)

## Bridge to `PoissonBracket`

`PoissonBracket(π)` exposes the same obstruction via `jacobi_obstruction()` and the textbook statement via `jacobi_condition()`. The killer move is `prove_jacobi_reduction(f, g, h)`, the cyclic Jacobi sum collapses to `[·,·]_SN(π, π)` in **one** proof step (the Derived Bracket Theorem).

In [ ]:
from jacopy.library.poisson import PoissonBracket

h = Symbol("h"); reg.declare(h, Graded(degree=-1))

P = PoissonBracket(pi)
print("obstruction:", P.jacobi_obstruction())
print("condition  :", P.jacobi_condition())

chain = P.prove_jacobi_reduction(f, g, h, registry=reg)
print(f"initial: {chain.initial}")
print(f"final  : {chain.final}")
print(f"steps  : {len(chain)}  rule: {chain.steps[0].rule}")

## When SN stays inert

Three situations:

1. **Atomic higher-order multivector**, useful opacity, the handle drives the proof.
2. **Symbolic SN-degree**, if any operand has a non-integer `Graded` degree, the wedge-Leibniz parity can't be decided.
3. **Forms**, `sn.expand(α, π)` for a form `α` is **not defined**. SN is the multivector-only bracket; the form-level operation is the Koszul bracket via `DerivedBracket(sn, π, acting_on=Sharp(π))` (tutorial 7).

## Summary

* `sn = SchoutenBracket()`, graded Lie bracket of degree 0 in the shifted grading `|X| = k − 1`.
* Four base cases close on 1-vector / function pairs; wedge Leibniz climbs into `Product`s.
* Atomic higher multivectors return an opaque `BracketApply`, `sn.self_bracket(π)` is the universal Poisson obstruction.
* `PoissonBracket.prove_jacobi_reduction` collapses the cyclic Jacobi sum to that obstruction in one step.